# MotherSettings Class

This notebook shows how the MotherSettings object can be used. Furthermore, the benefits are outlined. The MotherSettings object is based on [pydantic](https://docs.pydantic.dev/2.10/). First, it might be outlined that all provided transformers can be used without the `MotherSettings` class. However, it is highly recommended for input validation purposes.

In [22]:
from mother.settings import MotherSettings
import logging

logging.basicConfig(level=logging.INFO)

## Creation of an example settings file

To create an example settings file you can use the 'create' method of the MotherSettings class.

In [23]:
from pathlib import Path

settings_file_path: Path = Path.cwd().joinpath("mother_config.yaml")
MotherSettings.create(file_path=settings_file_path)

INFO:mother.settings:Creating yaml file at /workspaces/mother-ml/examples/notebooks/01_basics/mother_config.yaml


{
    "input": {
        "file": "examples/notebooks/freesolv_train.csv",
        "separator": ",",
        "structure_col": "smiles",
        "target_columns": [
            "expt"
        ],
        "group_col": "cat_col"
    },
    "pipeline": {
        "memory": null,
        "verbose": false,
        "transform": "pandas",
        "n_jobs": null,
        "remainder": "passthrough",
        "verbose_feature_names_out": false
    },
    "preprocessing": {
        "flags": [
            "STANDARDIZE",
            "NEUTRALIZE",
            "DESALT"
        ]
    },
    "feature_generation": {
        "fingerprints": [
            {
                "MorganFP": {
                    "radius": 2,
                    "fpSize": 1024,
                    "includeChirality": false
                }
            }
        ],
        "maccs": true,
        "chemical_descriptors": {
            "omit_prefixes": [
                "fr_",
                "FpDensity"
            ],
            "desc

In [24]:
settings_file_path

PosixPath('/workspaces/mother-ml/examples/notebooks/01_basics/mother_config.yaml')

In the dumped file you can see different sections with settings like 'preprocessing'. Each section can be used invdividiually for each transformer.
To load this file just use the `load_from_yaml` function.

In [25]:
ms = MotherSettings.load_from_yaml(settings_file_path)
ms.preprocessing

PreprocessingConfig(flags=['STANDARDIZE', 'NEUTRALIZE', 'DESALT'])

## Provide your own settings

To provide your own settings you can simply modify the dumped yaml file or you modify the settings object within the python code. In the following an example is shown how the `feature_generation` settings are modified bys disabling the `Maccs` features and using just a subset of the rdkit chemical descriptors.

In [26]:
ms.feature_generation

FeatureGenerationConfig(fingerprints=[{'MorganFP': {'fpSize': 1024, 'includeChirality': False, 'radius': 2}}], maccs=True, chemical_descriptors=ChemicalDescriptorsParams(omit_prefixes=('fr_', 'FpDensity'), descriptor_prefix='rdkit_', descriptor_list=None))

The simplest, but mostlikely error prone way is to define a string in yaml format that creates a dictionary containing the relevant settings.

In [27]:
import yaml

feature_generation = """
feature_generation:
  maccs: False
  fingerprints:
    - MorganFP:
        radius: 3
        fpSize: 256
        include_chirality: True
  chemical_descriptors:
    descriptor_prefix: "rdkit_"
    omit_prefixes: ["fr_"] #omit fragment features
    descriptor_list: ["MolWt","NumAromaticRings","NumHAcceptors","NumHDonors","NumHeteroatoms"] # subset of descriptors
"""
fg_settings = yaml.safe_load(feature_generation)
fg_settings

{'feature_generation': {'maccs': False,
  'fingerprints': [{'MorganFP': {'radius': 3,
     'fpSize': 256,
     'include_chirality': True}}],
  'chemical_descriptors': {'descriptor_prefix': 'rdkit_',
   'omit_prefixes': ['fr_'],
   'descriptor_list': ['MolWt',
    'NumAromaticRings',
    'NumHAcceptors',
    'NumHDonors',
    'NumHeteroatoms']}}}

In [28]:
fg_settings["feature_generation"]

{'maccs': False,
 'fingerprints': [{'MorganFP': {'radius': 3,
    'fpSize': 256,
    'include_chirality': True}}],
 'chemical_descriptors': {'descriptor_prefix': 'rdkit_',
  'omit_prefixes': ['fr_'],
  'descriptor_list': ['MolWt',
   'NumAromaticRings',
   'NumHAcceptors',
   'NumHDonors',
   'NumHeteroatoms']}}

In [29]:
from mother.feature_generation import config as fp_config

fg_conf: fp_config.FeatureGenerationConfig = fp_config.FeatureGenerationConfig(**fg_settings["feature_generation"])
fg_conf

FeatureGenerationConfig(fingerprints=[{'MorganFP': {'radius': 3, 'fpSize': 256, 'include_chirality': True}}], maccs=False, chemical_descriptors=ChemicalDescriptorsParams(omit_prefixes=('fr_',), descriptor_prefix='rdkit_', descriptor_list=['MolWt', 'NumAromaticRings', 'NumHAcceptors', 'NumHDonors', 'NumHeteroatoms']))